# 6. Audits

**Questions.** How complete is the service layer against the official registries, and is it right on the ground?

- **Registry match.** The share of DepEd public schools found in OpenStreetMap or Overture. The DOH facility registry cannot be exported, even in a browser, so health has no registry match; Healthsites.io, the open health cross-check, is used only when HDX lets it download.
- **Service audit.** A random 30 services within a 15-minute walk of each barangay's residents, to be checked on street imagery for found and open.
- **Walk audit.** 20 routes per barangay from street points inside it to the nearest service of each category in turn, to be scored on street imagery for sidewalk, crossings, shade and obstruction, with a count of sari-sari stores seen.

The two samples are fixed (seed 2026) worksheets with an imagery link per row and blank columns for the scores; judging the imagery is a by-hand step, so their results are not in this notebook.

In [1]:
import sys
from pathlib import Path

# The notebooks import their logic from src/: the shared helpers and the analysis functions.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir())
for folder in ("src/shared", "src/03_analyse"):
    sys.path.insert(0, str(ROOT / folder))

import measures
import pandas as pd
from common import TABLES

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
AREAS = measures.STUDY + ["Metro Manila"]
import audits
import walk_times

w, b = measures.load()
net = walk_times.Network()
svc = walk_times.load_services()

In [2]:
reg = audits.registry_match()
reg.to_csv(TABLES / "registry_match.csv", index=False, float_format="%.6g")
print("Healthsites.io check:", "used" if reg.attrs["healthsites"] else "not available (HDX bot challenge)")
reg.round(3)

Healthsites.io check: not available (HDX bot challenge)


,area,category,found,deped_schools,share_found
0,Metro Manila,public_elementary,410,501.0,0.818
1,Metro Manila,public_high,208,242.0,0.860
2,City of Manila,public_elementary,59,76.0,0.776
3,City of Manila,public_high,26,35.0,0.743
4,Quezon City,public_elementary,74,92.0,0.804
5,Quezon City,public_high,39,44.0,0.886
6,City of Pasig,public_elementary,26,29.0,0.897
7,City of Pasig,public_high,12,12.0,1.000
8,Loyola Heights,public_elementary,1,1.0,1.000
9,Loyola Heights,public_high,1,1.0,1.000


**Reading.** OpenStreetMap or Overture holds a record for 82% of Metro Manila's DepEd public elementary schools and 86% of its public high schools, from 74% (Manila high schools) to 100% (Pasig high schools); the DepEd schools in Loyola Heights and Oranbo are all found. Because DepEd defines the public-school categories, the misses lower the open sources' completeness, not the measures. There is no registry check for health: the DOH registry cannot be exported, and HDX still refuses the Healthsites.io download.

In [3]:
sample = audits.service_audit_sample(w, net, svc)
sample.to_csv(TABLES / "service_audit_sample.csv", index=False)
sample.groupby(["barangay", "category"]).size().unstack(fill_value=0)

category,food_retail,hospital,palengke,pharmacy,primary_care,public_elementary,public_high,transit
barangay,,,,,,,,
Barangay 136,5,2,1,3,4,4,1,10
Loyola Heights,15,0,0,3,2,1,0,9
Oranbo,13,1,1,4,7,0,0,4


In [4]:
routes = audits.walk_audit_routes(net, svc)
routes.to_csv(TABLES / "walk_audit_routes.csv", index=False)
routes.groupby("barangay").agg(routes=("route", "size"), median_m=("length_m", "median"), max_m=("length_m", "max"))

,routes,median_m,max_m
barangay,,,
Barangay 136,20,344.0,1008
Loyola Heights,20,616.0,1942
Oranbo,20,300.5,1226


**Reading.** The service audit sample holds 30 services per barangay drawn from those within 15 minutes of its residents, in proportion to what is there (food, transit and clinics most). The 60 walk-audit routes run from street points inside each barangay (within 200 m of Barangay 136, which has too few) to the nearest service of each category in turn, with medians of 300 to 620 m. Both worksheets wait for the by-hand imagery scoring.